# ST554_Project 2
Author: Huiping Zhou   
Data: 3/14/2026

# Part I-Creating a Class
In this section, we are going to create a class called `SparkDataCheck` and save it on a .py file. We will test the SparkDataCheck class on the [air](https://www4.stat.ncsu.edu/online/datasets/air.csv) dataset which read in as Spark SQL style data frames.

## Introduction

In [1]:
#import needed modules
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pandas.api.types import is_numeric_dtype
from pyspark.sql.types import StringType
from typing import Optional

In [2]:
# Create a SparkSession named 'my_app'
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 17:22:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/19 17:22:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


We read in dataset `air` using the method that created in class file. we import the my_class into this notebook first.

In [7]:
import my_class

In [5]:
#you can run this code and this will re-import that class that you have there without having to restart your kernel.
import importlib
importlib.reload(my_class)

<module 'my_class' from '/home/jupyter-hzhou13@ncsu.edu/Project2/my_class.py'>

## Reading in dataset using `from_spark` method
We use the `from_spark` method to read in the dataset and dispaly the first 5 rows using the `show()` method.

In [10]:
sql_df = my_class.SparkDataCheck.from_spark(spark, "air.csv", format="csv", header=True,inferSchema=True, sep=",")
sql_df = sql_df.drop("_c0")
sql_df.show(5)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|3/10/2004|2026-03-19 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-19 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|3/10/2004|2026-03-19 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-19 21:00:00|   2.2|       1376|      80|     9.2|        

Based on the output above, the `from_spark` method successfully read the data correctly.

In [11]:
#check the data types of clomuns
sql_df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08.S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08.S4(NO2): integer (nullable = true)
 |-- PT08.S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



We will update column names that include dots (.) to avoid potential problems when working with Spark DataFrames.

In [12]:
sql_air = (sql_df
        .withColumnRenamed('PT08.S1(CO)', 'S1_CO')
        .withColumnRenamed('PT08.S2(NMHC)', 'S2_NMHC')
        .withColumnRenamed('PT08.S3(NOx)', 'S3_NOx')
        .withColumnRenamed('PT08.S4(NO2)', 'S4_NO2')
        .withColumnRenamed('PT08.S5(O3)', 'S5_O3'))
sql_air.show(5)

+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888|
+-------

Since we know that this dataset uses -200 to code missing values, we will check whether -200 appears in any column by generating basic summary statistics using the `summary()` method.

In [16]:
num_cols = ['CO(GT)', 'S1_CO', 'NMHC(GT)', 'C6H6(GT)', 'S2_NMHC', 'NOx(GT)', 'S3_NOx', 'NO2(GT)',  'S4_NO2', 'S5_O3', 'T', 'RH', 'AH']
sql_air.select(*num_cols).summary('count', 'mean', 'stddev', 'min', 'max').show()

+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|            CO(GT)|             S1_CO|           NMHC(GT)|          C6H6(GT)|          S2_NMHC|          NOx(GT)|            S3_NOx|           NO2(GT)|            S4_NO2|             S5_O3|                T|               RH|                AH|
+-------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              9357|              9357|               9357|              9357|             9357|             9357|              9357|              9357|              9357|              9357|             9357|             9357|    

The basic summary statistics show that all numeric variables have a minimum value of -200, which is the sentinel value used in this dataset to represent missing observations. Then we will replace the coded missing values(-200) with NULL.

In [17]:
#replace the coded missing value (-200) with NULL in all numeric columns
sql_air =sql_air.na.replace({-200: None})
sql_air.show()

+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888|
|3/10/20

Next, we will show examples of calling each method from the my_class file on this object, beginning with 
### Testing the `check_numeric_range method`   
**Case 1**: Provide two bounds (2,5)   
When two bounds are provided, the `check_numeric_range` method appends a Boolean column named `new_CO(GT)` to the returned DataFrame.
- If the value in the selected column falls within the specified range, the method returns true.
- If the value is outside the range, it returns false.
- If the value is null, the method returns null in the new column.

In [18]:
# Create a SparkDataCheck instance from the existing sql_air DataFrame
sql_air_checker = my_class.SparkDataCheck(sql_air)

#case 1:provide two bounds (2,5)
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', lower = 2, upper = 5, new_column = 'new_CO(GT)')
check.select('Date', 'CO(GT)', 'new_CO(GT)').show(40)

+---------+------+----------+
|     Date|CO(GT)|new_CO(GT)|
+---------+------+----------+
|3/10/2004|   2.6|      true|
|3/10/2004|   2.0|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   1.6|     false|
|3/10/2004|   1.2|     false|
|3/11/2004|   1.2|     false|
|3/11/2004|   1.0|     false|
|3/11/2004|   0.9|     false|
|3/11/2004|   0.6|     false|
|3/11/2004|  NULL|      NULL|
|3/11/2004|   0.7|     false|
|3/11/2004|   0.7|     false|
|3/11/2004|   1.1|     false|
|3/11/2004|   2.0|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   1.7|     false|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.6|     false|
|3/11/2004|   1.9|     false|
|3/11/2004|   2.9|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.9|      true|
|3/11/2004|   4.8|      true|
|3/11/2004|   6.9|     false|
|3/11/2004|   6.1|     false|
|3/11/2004|   3.9|      true|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.0|     false|
|3/12/2004

The output above confirms that the results match our expectations. The `check_numeric_range` method behaves correctly for a numeric range with both lower and upper bounds.

**Case 2** Provide lower bound only  
When only one bound is provided (lower = 2), the method applies only to the lower side. Missing data will produce a NULL value in the Boolean result column. Any value greater ≥ 2 returns true.

In [19]:
#case 2 provide one bound
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', lower =2, new_column = 'new_CO(GT)')
check.select('Date', 'CO(GT)', 'new_CO(GT)').show(30)

+---------+------+----------+
|     Date|CO(GT)|new_CO(GT)|
+---------+------+----------+
|3/10/2004|   2.6|      true|
|3/10/2004|   2.0|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   1.6|     false|
|3/10/2004|   1.2|     false|
|3/11/2004|   1.2|     false|
|3/11/2004|   1.0|     false|
|3/11/2004|   0.9|     false|
|3/11/2004|   0.6|     false|
|3/11/2004|  NULL|      NULL|
|3/11/2004|   0.7|     false|
|3/11/2004|   0.7|     false|
|3/11/2004|   1.1|     false|
|3/11/2004|   2.0|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   1.7|     false|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.6|     false|
|3/11/2004|   1.9|     false|
|3/11/2004|   2.9|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.9|      true|
|3/11/2004|   4.8|      true|
|3/11/2004|   6.9|      true|
|3/11/2004|   6.1|      true|
|3/11/2004|   3.9|      true|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.0|     false|
+---------

The output above confirms that when only the lower bound is provided, the method evaluates the condition solely on that side. For rows with missing input values, the appended Boolean column correctly returns NULL, as intended.

**Case 3**: Provide only an upper bound (upper = 5)   
When only the upper bound is supplied, the method applies the range check only to the upper side. Values ≤ 5 return true while values > 5 return false. Missing values produce NULL in the Boolean column.

In [20]:
#case 2 provide only an upper bound
check=sql_air_checker.check_numeric_range(column = 'CO(GT)', upper =5, new_column = 'new')
check.select('Date', 'CO(GT)', 'new').show(30)

+---------+------+-----+
|     Date|CO(GT)|  new|
+---------+------+-----+
|3/10/2004|   2.6| true|
|3/10/2004|   2.0| true|
|3/10/2004|   2.2| true|
|3/10/2004|   2.2| true|
|3/10/2004|   1.6| true|
|3/10/2004|   1.2| true|
|3/11/2004|   1.2| true|
|3/11/2004|   1.0| true|
|3/11/2004|   0.9| true|
|3/11/2004|   0.6| true|
|3/11/2004|  NULL| NULL|
|3/11/2004|   0.7| true|
|3/11/2004|   0.7| true|
|3/11/2004|   1.1| true|
|3/11/2004|   2.0| true|
|3/11/2004|   2.2| true|
|3/11/2004|   1.7| true|
|3/11/2004|   1.5| true|
|3/11/2004|   1.6| true|
|3/11/2004|   1.9| true|
|3/11/2004|   2.9| true|
|3/11/2004|   2.2| true|
|3/11/2004|   2.2| true|
|3/11/2004|   2.9| true|
|3/11/2004|   4.8| true|
|3/11/2004|   6.9|false|
|3/11/2004|   6.1|false|
|3/11/2004|   3.9| true|
|3/11/2004|   1.5| true|
|3/11/2004|   1.0| true|
+---------+------+-----+
only showing top 30 rows


The output confirms that the method correctly evaluates the upper‑only condition and handles missing values as expected.

**Case 4**: No bound provided   
If no bounds are provided, a message is displayed reminding us that at least one bound is required, and the DataFrame is returned without modification.

In [21]:
# case 4: no bound provided
sql_air_checker.check_numeric_range(column = 'CO(GT)', new_column = 'new_CO(GT)').show(8)

Error: At least one of 'lower' or 'upper' must be provided.
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|     6.5|    836|  

The output above shows that this works as intended.

**Case 5**: provide a string column   
When a string column is supplied, the method issues a warning indicating that the selected column is not numeric. In this case, no validation is performed, and the method returns the original DataFrame without modification.

In [22]:
#case 5: provide a string colume
sql_air_checker.check_numeric_range(column = 'Date').show(8)

+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888|
|3/10/20

The output above verifies that the method handles string columns correctly by preventing numeric validation and leaving the DataFrame unchanged.

Across the four test scenarios (1) providing both bounds, (2) providing a single lower bound, (3) providing a single upper bound, (4) providing no bounds, and (4) providing a string column, the method behaved exactly as expected. These results confirm that the `check_numeric_range` method is implemented correctly and handles all tested input conditions as intended.

### Testing `check_string_levels` method
Since the dataset does not contain any string variables with missing values, we create a binary variable `C6H6_label` from the numeric column `C6H6(GT)`. Benzene concentration (C6H6(GT)) is a key indicator of outdoor air quality. When the concentration is ≤ 5, the air quality is considered good; otherwise, it is considered poor. Therefore, we define C6H6(GT) ≤ 5 as `Good` and values greater than 5 as `Bad`.

In [23]:
#Step 1: Create a string/categorical column from numeric quality
#map to 'good' if <= 5 else 'bad'; preserve NaN as NaN
df_string = sql_air.withColumn(
    "C6H6_label",
    F.when(F.col("C6H6(GT)").isNull(), F.lit(None).cast(StringType()))
    .when(F.col("C6H6(GT)") <= 5, F.lit("Good"))
    .otherwise(F.lit("Bad"))
)
df_string.show(10)

+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|       Bad|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|  

In above table, I created a binary variable called `C6H6_label` based on the numeric variable `C6H6(GT)`. I defined concentrations ≤ 5 as good air quality, and values above 5 as bad air quality.

**Case 1**: The binary varibale `C6H6_label` is used to test `check_string_levels` method; We will check the levels = Good.   
In this test, the binary string column `C6H6_label` is used to evaluate the behavior of the `check_string_levels` method. We specify the expected level as `Good`, and the method will check whether each entry in the column matches this user‑defined set of levels.

In [24]:
# first create an instance of your SparkDataCheck class using your df DataFrame
df_string_checker = my_class.SparkDataCheck(df_string)

#case 1: check the levels = good
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = ['Good'], new_column="is_good")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'is_good').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("is_good").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'is_good').show(10)

+---------+--------+----------+-------+
|     Date|C6H6(GT)|C6H6_label|is_good|
+---------+--------+----------+-------+
|3/10/2004|    11.9|       Bad|  false|
|3/10/2004|     9.4|       Bad|  false|
|3/10/2004|     9.0|       Bad|  false|
|3/10/2004|     9.2|       Bad|  false|
|3/10/2004|     6.5|       Bad|  false|
|3/10/2004|     4.7|      Good|   true|
|3/11/2004|     3.6|      Good|   true|
|3/11/2004|     3.3|      Good|   true|
|3/11/2004|     2.3|      Good|   true|
|3/11/2004|     1.7|      Good|   true|
+---------+--------+----------+-------+
only showing top 10 rows
+--------+--------+----------+-------+
|    Date|C6H6(GT)|C6H6_label|is_good|
+--------+--------+----------+-------+
|4/1/2004|    NULL|      NULL|   NULL|
|4/1/2004|    NULL|      NULL|   NULL|
|4/1/2004|    NULL|      NULL|   NULL|
|4/8/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      NULL|   NULL|
|4/9/2004|    NULL|      

The results confirms that when each value in the string column `C6H6_label` is compared against the the user-specified set of levels (Good), the `check_string_levels` method behaves correctly. The method appends a Boolean column named `is_good`, where rows labeled *Good* return true and rows labeled *Bad* return false. Additionally, the results show that when `C6H6(GT)` is NULL, both `C6H6_label` and the derived `is_good` column are also NULL. Overall, the method performs as intended for this scenario.

**Case 2**: Check the levels = `Bad`  
In this case, we test the `check_string_levels` method by specifying `Bad` as the expected level. When `Bad` is provided as the target level, the method should return **true** for rows where C6H6_label is `Bad` and **false** for rows labeled `Good`. Missing values should produce **NULL** in the resulting Boolean column.

In [25]:
#case 2: cehck the levels = Bad
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = ['Bad'], new_column="is_bad")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'is_bad').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("is_bad").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'is_bad').show(5)

+---------+--------+----------+------+
|     Date|C6H6(GT)|C6H6_label|is_bad|
+---------+--------+----------+------+
|3/10/2004|    11.9|       Bad|  true|
|3/10/2004|     9.4|       Bad|  true|
|3/10/2004|     9.0|       Bad|  true|
|3/10/2004|     9.2|       Bad|  true|
|3/10/2004|     6.5|       Bad|  true|
|3/10/2004|     4.7|      Good| false|
|3/11/2004|     3.6|      Good| false|
|3/11/2004|     3.3|      Good| false|
|3/11/2004|     2.3|      Good| false|
|3/11/2004|     1.7|      Good| false|
+---------+--------+----------+------+
only showing top 10 rows
+--------+--------+----------+------+
|    Date|C6H6(GT)|C6H6_label|is_bad|
+--------+--------+----------+------+
|4/1/2004|    NULL|      NULL|  NULL|
|4/1/2004|    NULL|      NULL|  NULL|
|4/1/2004|    NULL|      NULL|  NULL|
|4/8/2004|    NULL|      NULL|  NULL|
|4/9/2004|    NULL|      NULL|  NULL|
+--------+--------+----------+------+
only showing top 5 rows


The results indicate that when each value in the string column C6H6_label matches the user-specified level (`Bad`), the `check_string_levels` method correctly returns the DataFrame with an appended Boolean column `is_bad`. Values labeled `Bad` in the `C6H6_label` column return true in the `is_bad` column, while values labeled `Good` return false. The output also shows that when `C6H6(GT)` is NULL, both `C6H6_label` and the derived `is_bad` column are also NULL. Overall, the method is functioning as intended for this scenario.

**Case 3**: No string column is provided   
In this case, we intentionally select a column that is not of string type to test how the `check_string_levels` method handles invalid input. The method should detect that the column is not a string column, issue a warning, and return the original DataFrame without making any changes.

In [26]:
#case 3: no string column is provided
check = df_string_checker.check_string_levels(column = "S1_CO", levels = ['Good'], new_column="is_good")
check.show(5)

+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|       Bad|
|3/10/2004|2026-03-19 22:00:00|   1.6| 1272|      51|  

The results show that a warning is issued indicating that the selected column is not of string type, and the method returns the DataFrame without any modifications. The `check_string_levels` method behaves as intended for this scenario.

**Case 4**: No levels are provided (empty list)  
In this case, we intentionally supply an empty list of allowed levels to test how the `check_string_levels` method behaves when the user provides no valid levels for comparison. The method should warn that the list of levels is empty and then proceed with the default behavior.

In [27]:
#case 4: no levels are provided
check = df_string_checker.check_string_levels(column = "C6H6_label", levels = [], new_column="new")
check.select('Date', 'C6H6(GT)', 'C6H6_label', 'new').show(10)
# Filter for rows where any of the specified columns are null and select them
check.filter(
    F.col("C6H6(GT)").isNull() | F.col("C6H6_label").isNull() | F.col("new").isNull()
).select('Date', 'C6H6(GT)', 'C6H6_label', 'new').show(5)

+---------+--------+----------+-----+
|     Date|C6H6(GT)|C6H6_label|  new|
+---------+--------+----------+-----+
|3/10/2004|    11.9|       Bad|false|
|3/10/2004|     9.4|       Bad|false|
|3/10/2004|     9.0|       Bad|false|
|3/10/2004|     9.2|       Bad|false|
|3/10/2004|     6.5|       Bad|false|
|3/10/2004|     4.7|      Good|false|
|3/11/2004|     3.6|      Good|false|
|3/11/2004|     3.3|      Good|false|
|3/11/2004|     2.3|      Good|false|
|3/11/2004|     1.7|      Good|false|
+---------+--------+----------+-----+
only showing top 10 rows
+--------+--------+----------+----+
|    Date|C6H6(GT)|C6H6_label| new|
+--------+--------+----------+----+
|4/1/2004|    NULL|      NULL|NULL|
|4/1/2004|    NULL|      NULL|NULL|
|4/1/2004|    NULL|      NULL|NULL|
|4/8/2004|    NULL|      NULL|NULL|
|4/9/2004|    NULL|      NULL|NULL|
+--------+--------+----------+----+
only showing top 5 rows


The results show that when an empty list of levels is provided, the method issues a warning indicating that the levels argument is empty and returns the DataFrame with an appended column `new`. Regardless of whether the value in `C6H6_label` is `Good` or `Bad`, the new column correctly returns **false** for all non‑NULL entries. In addition, the output confirms that whenever `C6H6(GT)` is NULL, both `C6H6_label` and the derived `new` column are also **NULL**. Overall, the method behaves correctly for this scenario.

**Case 5**: Column is not in the dataset   
In this case, we intentionally provide an invalid (non‑existent) column name to test whether the `check_string_levels` method can correctly validate the input. When the specified column is not found in the DataFrame, the method should issue a warning and return the original DataFrame without making any modifications.

In [28]:
#Case 5: column is not in the dataset
check = df_string_checker.check_string_levels(column = "beneze", levels = ['Good'], new_column="new")
check.show(5)

Error: Column 'beneze' not found in DataFrame. Returning DataFrame without modification.
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 

The results indicate that when a column name that does not exist in the DataFrame is provided, the method prints a warning to inform the user that the variable is not present in the dataset. As expected, the method then returns the original DataFrame unchanged. This confirms that the `check_string_levels method` properly handles invalid column references.

Overall, the results across all five test cases confirm that the `check_string_levels` method is implemented correctly and behaves as expected under all tested conditions.

### Testing the `check_missing` method
**Case 1**: Check if the cloumn `C6H6(GT)` has missing values   
We apply the `check_missing` method to the column `C6H6(GT)`, which generates a new Boolean column named `missing`. This column contains **true** when a value in C6H6(GT) is NULL, and **false** otherwise.

In [29]:
#case 1: Check column C6H6(GT)
check = df_string_checker.check_missing(column = "C6H6(GT)", new_column = 'missing')
check.select('Date', 'C6H6(GT)', 'missing').show(10)
#show the first 10 rows where the 'missing'column will labeled true
check.filter("missing = true") \
     .select("Date", "C6H6(GT)", "missing") \
     .show(10)

+---------+--------+-------+
|     Date|C6H6(GT)|missing|
+---------+--------+-------+
|3/10/2004|    11.9|  false|
|3/10/2004|     9.4|  false|
|3/10/2004|     9.0|  false|
|3/10/2004|     9.2|  false|
|3/10/2004|     6.5|  false|
|3/10/2004|     4.7|  false|
|3/11/2004|     3.6|  false|
|3/11/2004|     3.3|  false|
|3/11/2004|     2.3|  false|
|3/11/2004|     1.7|  false|
+---------+--------+-------+
only showing top 10 rows
+--------+--------+-------+
|    Date|C6H6(GT)|missing|
+--------+--------+-------+
|4/1/2004|    NULL|   true|
|4/1/2004|    NULL|   true|
|4/1/2004|    NULL|   true|
|4/8/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
|4/9/2004|    NULL|   true|
+--------+--------+-------+
only showing top 10 rows


The output aligns with our expectations, confirming that the `check_missing` method functions correctly for the `C6H6(GT)` column.

**Case 2**: Check column `CO(GT)`   
Similarly, we apply the `check_missing` method to the `CO(GT)` column. As in Case 1, the method should correctly append a Boolean column indicating whether each value in CO(GT) is NULL.

In [30]:
#Case2: check column CO(GT)
check = df_string_checker.check_missing(column = "CO(GT)", new_column = 'missing')
check.select('Date', 'CO(GT)', 'missing').show(10)
#show the first 10 rows where the 'missing'column will labeled true
check.filter("missing = true") \
     .select("Date", "CO(GT)", "missing") \
     .show(10)

+---------+------+-------+
|     Date|CO(GT)|missing|
+---------+------+-------+
|3/10/2004|   2.6|  false|
|3/10/2004|   2.0|  false|
|3/10/2004|   2.2|  false|
|3/10/2004|   2.2|  false|
|3/10/2004|   1.6|  false|
|3/10/2004|   1.2|  false|
|3/11/2004|   1.2|  false|
|3/11/2004|   1.0|  false|
|3/11/2004|   0.9|  false|
|3/11/2004|   0.6|  false|
+---------+------+-------+
only showing top 10 rows
+---------+------+-------+
|     Date|CO(GT)|missing|
+---------+------+-------+
|3/11/2004|  NULL|   true|
|3/12/2004|  NULL|   true|
|3/12/2004|  NULL|   true|
|3/13/2004|  NULL|   true|
|3/14/2004|  NULL|   true|
|3/15/2004|  NULL|   true|
|3/16/2004|  NULL|   true|
|3/17/2004|  NULL|   true|
|3/18/2004|  NULL|   true|
|3/19/2004|  NULL|   true|
+---------+------+-------+
only showing top 10 rows


The output meets our expectations, confirming that the method `check_missing` works correctly for column `CO(GT)`.

**Case 3**: Check the `Date` column   
We check whether there are any missing values in the `Date` column.

In [31]:
#Case 3: check the date column
check = df_string_checker.check_missing(column = 'Date', new_column = 'missing')
check.select('Time', 'Date', 'missing').show(10)
#show the first 10 rows where the 'missing'column will labeled true
check.filter("missing = true") \
     .select("Time", "Date", "missing") \
     .show(10)

+-------------------+---------+-------+
|               Time|     Date|missing|
+-------------------+---------+-------+
|2026-03-19 18:00:00|3/10/2004|  false|
|2026-03-19 19:00:00|3/10/2004|  false|
|2026-03-19 20:00:00|3/10/2004|  false|
|2026-03-19 21:00:00|3/10/2004|  false|
|2026-03-19 22:00:00|3/10/2004|  false|
|2026-03-19 23:00:00|3/10/2004|  false|
|2026-03-19 00:00:00|3/11/2004|  false|
|2026-03-19 01:00:00|3/11/2004|  false|
|2026-03-19 02:00:00|3/11/2004|  false|
|2026-03-19 03:00:00|3/11/2004|  false|
+-------------------+---------+-------+
only showing top 10 rows
+----+----+-------+
|Time|Date|missing|
+----+----+-------+
+----+----+-------+



The output indicates that there are no missing values in the `Date` column.

**Case 4**: Provide an incorrect column name    
When a column name that does not exist in the dataset is supplied, the method outputs a warning indicating that the specified column cannot be found. In this situation, it returns the original DataFrame unchanged.

In [32]:
#case 4 provide a wrong column name
check = df_string_checker.check_missing(column = 'benzene', new_column = 'missing')
check.show(5)

Error: Column 'benzene' not found in DataFrame. Returning DataFrame without modification.
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|     Date|               Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|C6H6_label|
+---------+-------------------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+----------+
|3/10/2004|2026-03-19 18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|       Bad|
|3/10/2004|2026-03-19 19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|       Bad|
|3/10/2004|2026-03-19 20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|       Bad|
|3/10/2004|2026-03-19 21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584|

The output meets our expectations, indicating that the method functions correctly and has been implemented successfully.

Across all four scenarios, the `check_missing` method performs as expected. It correctly identifies missing values, handles valid and invalid column names appropriately, and behaves consistently. This confirms that the method is functioning properly and has been successfully implemented.

### Testing the `count_min_max` method
We defined two methods to do some statistic summarise for the data. We will test the `count_min_max` method first.

We want to report the minimum and maximum values of numeric columns by group. To do this, we first create a string column for `season` by mapping each Date value to one of the four seasons. We then save the resulting dataset as `df3` and use this dataset to test the functionality of the `count_min_max` method.

In [33]:
# 1) Parse the string to a DateType column
# - 'M/d/yyyy' handles single-digit months/days
df1 = sql_air.withColumn("parsed_date", F.to_date(F.col("Date"), "M/d/yyyy"))

# 2) Derive month (1-12)
df2 = df1.withColumn("month_num", F.month("parsed_date"))

# 3) Map month to season
df3 = df2.withColumn(
    "season",
    F.when(F.col("month_num").isin(12, 1, 2), F.lit("Winter"))
     .when(F.col("month_num").isin(3, 4, 5), F.lit("Spring"))
     .when(F.col("month_num").isin(6, 7, 8), F.lit("Summer"))
     .when(F.col("month_num").isin(9, 10, 11), F.lit("Fall"))
     .otherwise(F.lit(None))  # stays NULL if date couldn't be parsed
)
# Corrected: Pass column names as separate arguments to .drop()
df3 = df3.drop('Time',"month_num", 'parsed_date')
df3.show(5)

+---------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+------+
|     Date|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|season|
+---------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+------+
|3/10/2004|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|Spring|
|3/10/2004|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|Spring|
|3/10/2004|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|Spring|
|3/10/2004|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|Spring|
|3/10/2004|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888|Spring|
+---------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+------+
o

**Case 1**: No column is provided   
If no coloumns suplied, the `count_min_max` method will report the minimum and maximum values for all numerics and return it to a pdndas data frame.

In [38]:
#create an instance of the SparkDataCheck class using df DataFrame
df3_check = my_class.SparkDataCheck(df3)
#case 1: no column is specificed
check = df3_check.count_min_max()

In [39]:
#The check object from the previous output contains min/max for all numeric columns in a single row.
#We will transform it to have one variable per row.
result_list = []
for col in check.columns:
    if col.startswith('min_'):
        variable_name = col.replace('min_', '')
        max_col_name = 'max_' + variable_name
        if max_col_name in check.columns:
            min_val = check[col].iloc[0]
            max_val = check[max_col_name].iloc[0]
            result_list.append({'Variable': variable_name, 'Min': min_val, 'Max': max_val})

df_reshaped = pd.DataFrame(result_list)
display(df_reshaped)

,Variable,Min,Max
0,CO(GT),0.1000,11.900
1,S1_CO,647.0000,2040.000
2,NMHC(GT),7.0000,1189.000
3,C6H6(GT),0.1000,63.700
4,S2_NMHC,383.0000,2214.000
5,NOx(GT),2.0000,1479.000
6,S3_NOx,322.0000,2683.000
7,NO2(GT),2.0000,340.000
8,S4_NO2,551.0000,2775.000
9,S5_O3,221.0000,2523.000


The table above shows that the method `count_min_max` correctly reports the minimum and maximum values for all numeric variables. The method count_min_max meets our requirements.

**Case 2**: No column is specified and the data is grouped by season   
In this case, the method reports the minimum and maximum values for all numeric variables grouped by season and returns the results as a pandas DataFrame.

In [40]:
#case 2: No column is provided and the data is grouped by season
check = df3_check.count_min_max(group_by_col = 'season')
check

,season,min_CO(GT),max_CO(GT),min_S1_CO,max_S1_CO,min_NMHC(GT),max_NMHC(GT),min_C6H6(GT),max_C6H6(GT),min_S2_NMHC,...,min_S4_NO2,max_S4_NO2,min_S5_O3,max_S5_O3,min_T,max_T,min_RH,max_RH,min_AH,max_AH
0,Fall,0.1,11.9,647,2008,NaN,NaN,0.2,63.7,397,...,697,2775,261,2522,1.3,40.7,13.8,87.2,0.1988,2.2310
1,Spring,0.1,8.1,715,2040,7.0,1189.0,0.2,40.3,387,...,551,2684,221,2359,-1.9,32.8,9.2,85.2,0.1847,1.6296
2,Summer,0.1,6.4,708,1626,NaN,NaN,0.5,37.3,437,...,1084,2746,334,2475,16.2,44.6,9.6,82.3,0.6591,2.1806
3,Winter,0.1,9.9,691,1881,NaN,NaN,0.1,50.8,383,...,621,2405,252,2523,0.3,20.3,16.3,88.7,0.2269,1.3713


The output above reports the minimum and maximum values for all numeric variables by season. The method `count_min_max` works accurately.

**Case 3**: Provide one numeric column `C6H6(GT)` and grouped by `season`   

When checking a numeric variable with a grouping column, the `count_min_max` method report the minimum and maximum values of C6H6(GT) for each season and returns the results as a pandas DataFrame.

In [41]:
#case 3: Check C6H6(GT) with grouping
check = df3_check.count_min_max(column = 'C6H6(GT)', group_by_col = 'season')
check

,season,min_C6H6(GT),max_C6H6(GT)
0,Spring,0.2,40.3
1,Summer,0.5,37.3
2,Fall,0.2,63.7
3,Winter,0.1,50.8


The output above shows the minimum and maximum values of `C6H6(GT)` for each season. Fall shows the greatest variability with the highest peak value (63.7), while Winter records the lowest minimum (0.1). Summer has comparatively narrower variation, starting at a higher minimum (0.5). The `count_min_max` method works correctly.

**Case 4**: only provide one numeric column `C6H6(GT)`   
In this scenario, the method is given a single numeric column, C6H6(GT). The method processes it by reporting the minimum and maximum values for that column.

In [42]:
#case 4: check C6H6(GT) 
check = df3_check.count_min_max(column = 'C6H6(GT)')
check

,min_C6H6(GT),max_C6H6(GT)
0,0.1,63.7


The output shows a minimum value of 0.1 and a maximum value of 63.7 for C6H6(GT), which aligns with our expectations. This confirms that the method behaves correctly in the single‑column case.

**Case 5**: provide a string   
When a non‑numeric (string) column is supplied, the method prints a message indicating that the column is not numeric and returns None as expected.

In [43]:
#case 5: check Date 
check = df3_check.count_min_max(column = 'Date')
check

A warning is correctly displayed, reminding us that an invalid column type was provided. The method behaves as intended and meets our expectations.

Across all five scenarios, the `count_min_max` method behaves exactly as intended. It correctly handles numeric columns, identifies missing values, responds appropriately to invalid or non‑numeric input, and produces reliable outputs. These results confirm that the method has been implemented successfully.

### Testing the `counts_string` method
We use the `df_string` dataset generated during the `testing of the check_string_levels method`, as it contains the binary variable `C6H6_label`, which was created from the original `C6H6(GT)` variable.

In [44]:
#check the type of variables
df_string.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- S1_CO: integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- S2_NMHC: integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- S3_NOx: integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- S4_NO2: integer (nullable = true)
 |-- S5_O3: integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- C6H6_label: string (nullable = true)



In the df_string dataset, only `Date and C6H6_label` are string variables, so we use these two columns to test the `counts_string` method.

**Case 1**: Check a numeric variable `CO(GT)`   
When a numeric variable is supplied, the method should correctly identify that the input is of the wrong type. In this situation, it must issue a warning and avoid performing any calculations.

In [45]:
#create an instance of the SparkDataCheck class using df DataFrame
counts_check = my_class.SparkDataCheck(df_string)

#case 1: Check a numeric variable
check = counts_check.counts_string(col1 = 'CO(GT)')

Column 'CO(GT)' is not a string.


A warning was issued indicating that the column `CO(GT)` is not a string type, and the method returned None as expected.

**Case 2**: Provide a string variable `Date`  
When a valid string variable is supplied, the method should correctly identify it as the appropriate type and proceed to report the counts for each level of the `Date` variable.

In [37]:
#case 2: check Date variable
check = counts_check.counts_string(col1 = 'Date').orderBy('Date')
check.show()

+---------+-----+
|     Date|count|
+---------+-----+
| 1/1/2005|   24|
|1/10/2005|   24|
|1/11/2005|   24|
|1/12/2005|   24|
|1/13/2005|   24|
|1/14/2005|   24|
|1/15/2005|   24|
|1/16/2005|   24|
|1/17/2005|   24|
|1/18/2005|   24|
|1/19/2005|   24|
| 1/2/2005|   24|
|1/20/2005|   24|
|1/21/2005|   24|
|1/22/2005|   24|
|1/23/2005|   24|
|1/24/2005|   24|
|1/25/2005|   24|
|1/26/2005|   24|
|1/27/2005|   24|
+---------+-----+
only showing top 20 rows


The output shows that every distinct date has a count of 24, indicating that the dataset contains hourly measurements. This confirms that the `count_string` method performed as expected for a valid string input.

**Case 3**: Check string variable `C6H6_label`   
Similar to Case 2, when a valid string variable `C6H6_label` is provided, the method should identify it as a string type and return the counts for each level.

In [46]:
#case3 check C6H6_label
check = counts_check.counts_string(col1='C6H6_label')
check.show()

+----------+-----+
|C6H6_label|count|
+----------+-----+
|      NULL|  366|
|      Good| 2607|
|       Bad| 6384|
+----------+-----+



The method reports 2,607 records classified as good air quality and 6,384 as bad, with 366 missing values. These results confirm that the `counts_string` method is functioning as intended.

**Case 4**: Provide two string variables  
When two string variables are supplied, the method should correctly identify both as valid string inputs and produce counts grouped by the combinations of the two variables.

In [47]:
#case 4: two string variables
check = counts_check.counts_string(col1 = 'Date', col2='C6H6_label').orderBy('Date')
check.show(50)

+---------+----------+-----+
|     Date|C6H6_label|count|
+---------+----------+-----+
| 1/1/2005|       Bad|   18|
| 1/1/2005|      Good|    6|
|1/10/2005|       Bad|   19|
|1/10/2005|      Good|    5|
|1/11/2005|      Good|    6|
|1/11/2005|       Bad|   18|
|1/12/2005|       Bad|   21|
|1/12/2005|      Good|    3|
|1/13/2005|      Good|    3|
|1/13/2005|       Bad|   21|
|1/14/2005|       Bad|   20|
|1/14/2005|      Good|    4|
|1/15/2005|      Good|    4|
|1/15/2005|       Bad|   20|
|1/16/2005|      Good|   14|
|1/16/2005|       Bad|   10|
|1/17/2005|       Bad|   16|
|1/17/2005|      Good|    8|
|1/18/2005|      Good|    8|
|1/18/2005|       Bad|   16|
|1/19/2005|       Bad|    9|
|1/19/2005|      Good|   15|
| 1/2/2005|       Bad|   11|
| 1/2/2005|      NULL|    3|
| 1/2/2005|      Good|   10|
|1/20/2005|       Bad|   16|
|1/20/2005|      Good|    8|
|1/21/2005|       Bad|   22|
|1/21/2005|      Good|    2|
|1/22/2005|       Bad|   24|
|1/23/2005|      Good|    9|
|1/23/2005|   

When both `Date and C6H6_label` were supplied as string variables, the method successfully generated counts grouped by the combination of these two columns. The output shows that for each individual date, the method reported how many records fell into each air‑quality label (Good, Bad, or NULL). Most dates contained only Good records (all 24 hourly observations), while a few dates included a mixture of Good, Bad, and NULL values.This demonstrates that the method correctly.

Across all four cases, the `count_string` method consistently identified valid string variables and correctly produced record counts for each level of the provided string column(s). These results confirm that the method has been successfully implemented.

**Case 5**: Provide one numeric column and one string column   
When a numeric and a string column are given, the method will detect that the numeric column is not a string, issue a warning, ignore the numeric column, and return the counts for the string column only.

In [48]:
#case 5: one numeric and one string column
check = counts_check.counts_string(col1 = 'C6H6(GT)', col2='C6H6_label')
check.show(50)

+----------+-----+
|C6H6_label|count|
+----------+-----+
|      NULL|  366|
|      Good| 2607|
|       Bad| 6384|
+----------+-----+



The output now shows that despite providing a numeric column (`C6H6(GT)`), the method successfully processed the valid string column (`C6H6_label`) and returned its counts, along with a warning for the ignored numeric column. This demonstrates the `counts_string` method functions as expected when one string and one numeric variabl are provided.


Across all four cases, the `count_string` method consistently identified valid string variables and correctly produced record counts for each level of the provided string column(s). These results confirm that the method has been successfully implemented.

## Import `air` dataset via Pandas
First, we read the `air` data set in using pandas (not pandas-on-spark). 

In [54]:
#read in data and drop the Unnamed: 0 column
pdf = pd.read_csv("air.csv", delimiter = ",")
pdf = pdf.drop('Unnamed: 0', axis=1, errors='ignore')
pdf.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


### Convert to Spark DataFrame
We use our own definde the `from_pandas` method to convert the padans DataFrame (pdf) to a Spark DataFrame.

In [55]:
#convert from padans df(pdf)
df_pandas = my_class.SparkDataCheck.from_pandas(spark, pdf) #convert from padans df(pdf)
df_pandas.show()

+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|          948|    172|        1092|    122|        1584|       1203|11.0|60.0|0.7867|

### Data Cleaning
We will replace all coded missing values (−200) with NULL and also remove all dots (.) from the column names. 

In [57]:
df_pandas = (df_pandas.na.replace({-200: None})
        .withColumnRenamed('PT08.S1(CO)', 'S1_CO')
        .withColumnRenamed('PT08.S2(NMHC)', 'S2_NMHC')
        .withColumnRenamed('PT08.S3(NOx)', 'S3_NOx')
        .withColumnRenamed('PT08.S4(NO2)', 'S4_NO2')
        .withColumnRenamed('PT08.S5(O3)', 'S5_O3'))
df_pandas.show()

+---------+--------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|     Date|    Time|CO(GT)|S1_CO|NMHC(GT)|C6H6(GT)|S2_NMHC|NOx(GT)|S3_NOx|NO2(GT)|S4_NO2|S5_O3|   T|  RH|    AH|
+---------+--------+------+-----+--------+--------+-------+-------+------+-------+------+-----+----+----+------+
|3/10/2004|18:00:00|   2.6| 1360|     150|    11.9|   1046|    166|  1056|    113|  1692| 1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0| 1292|     112|     9.4|    955|    103|  1174|     92|  1559|  972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2| 1402|      88|     9.0|    939|    131|  1140|    114|  1555| 1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2| 1376|      80|     9.2|    948|    172|  1092|    122|  1584| 1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6| 1272|      51|     6.5|    836|    131|  1205|    116|  1490| 1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2| 1197|      38|     4.7|    750|     89|  1337|     96|  1393|  949|1

### Testing the `check_numeric_range`method
Provide two bounds (2,5) on `CO(GT)` variable     
When two bounds are provided, the `check_numeric_range` method appends a Boolean column named `new_CO(GT)` to the returned DataFrame. If the value in the selected column falls within the specified range, the method returns true. If the value is outside the range, it returns false. If the value is null, the method returns null in the new column.

In [59]:
# Create a SparkDataCheck instance from the existing sql_air DataFrame
checker = my_class.SparkDataCheck(df_pandas)

# provide two bounds (2, 5)
check=checker.check_numeric_range(column = 'CO(GT)', lower = 2, upper = 5, new_column = 'new_CO(GT)')
check.select('Date', 'CO(GT)', 'new_CO(GT)').show(40)

+---------+------+----------+
|     Date|CO(GT)|new_CO(GT)|
+---------+------+----------+
|3/10/2004|   2.6|      true|
|3/10/2004|   2.0|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   2.2|      true|
|3/10/2004|   1.6|     false|
|3/10/2004|   1.2|     false|
|3/11/2004|   1.2|     false|
|3/11/2004|   1.0|     false|
|3/11/2004|   0.9|     false|
|3/11/2004|   0.6|     false|
|3/11/2004|  NULL|      NULL|
|3/11/2004|   0.7|     false|
|3/11/2004|   0.7|     false|
|3/11/2004|   1.1|     false|
|3/11/2004|   2.0|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   1.7|     false|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.6|     false|
|3/11/2004|   1.9|     false|
|3/11/2004|   2.9|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.2|      true|
|3/11/2004|   2.9|      true|
|3/11/2004|   4.8|      true|
|3/11/2004|   6.9|     false|
|3/11/2004|   6.1|     false|
|3/11/2004|   3.9|      true|
|3/11/2004|   1.5|     false|
|3/11/2004|   1.0|     false|
|3/12/2004

The output indicates that the method successfully appended the column `new_CO(GT)`, correctly returning true for values within the specified range (2, 5), false for values outside the range, and NULL for missing values. This confirms that the method functions as intended on dataset reading in using pandas and covert it to pandas-on-spark dataframe using our defined method `from_pandas` .

# Part II